# Module 02 — Supervised Learning
## 02-02: Logistic Regression & Binary Classification

**Objective:** Derive logistic regression from Bernoulli maximum likelihood, implement gradient descent with L1 and L2 regularization from scratch, visualize how different constraint geometries produce sparsity vs. shrinkage, and build a complete evaluation suite including ROC and AUC.

**Prerequisites:** 2-01 (Linear Regression), 1-07 (Probability & Statistics)

## Part 0 — Setup & Prerequisites

This notebook derives logistic regression from first principles. The sigmoid activation emerges from fitting a Bernoulli likelihood via MLE, binary cross-entropy is its negative log-likelihood, and gradient descent follows directly from differentiating that loss. We implement full-batch gradient descent with both L1 (Lasso) and L2 (Ridge) regularization — L1 uses the proximal gradient method (soft thresholding) to handle the non-differentiable penalty — and evaluate the models thoroughly on `sklearn.make_moons`.

**Prerequisites:** 2-01 (Linear Regression), 1-07 (Probability & Statistics)

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D

from sklearn.datasets import make_moons, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import (
    confusion_matrix as sk_confusion_matrix,
    roc_curve as sk_roc_curve,
    auc as sk_auc,
)

np.set_printoptions(precision=4, suppress=True)
print(f'Python : {sys.version.split()[0]}')
print(f'NumPy  : {np.__version__}')

In [ ]:
# ── Constants & Configuration ─────────────────────────────────────────────────
SEED = 1103
N_SAMPLES = 1200
NOISE = 0.25
TEST_SIZE = 0.20
LEARNING_RATE = 0.1
NUM_EPOCHS = 3000
TOLERANCE = 1e-8
C_DEFAULT = 1.0

# Regularization sweep for path analysis
C_VALUES = [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]

COLORS = ['#2196F3', '#F44336']   # class 0 = blue, class 1 = red
LABELS = ['Class 0', 'Class 1']

rng = np.random.default_rng(seed=SEED)
np.random.seed(SEED)

print(f'SEED={SEED} | N={N_SAMPLES} | noise={NOISE} | test_size={TEST_SIZE:.0%}')
print(f'LR={LEARNING_RATE} | epochs={NUM_EPOCHS} | tolerance={TOLERANCE}')

In [ ]:
# ── Dataset: sklearn make_moons ───────────────────────────────────────────────
X_raw, y = make_moons(n_samples=N_SAMPLES, noise=NOISE, random_state=SEED)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

# Standardize — gradient descent converges faster on unit-scaled inputs
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f'make_moons | noise={NOISE} | seed={SEED}')
print(f'  Full dataset : {X_raw.shape[0]:,} samples, {X_raw.shape[1]} features')
print(f'  Train split  : {X_train.shape[0]:,} samples (80 %)')
print(f'  Test split   : {X_test.shape[0]:,} samples  (20 %)')
print()
print('Class distribution (train):')
classes, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(classes, counts):
    print(f'  Class {cls}: {cnt:,} samples ({cnt / len(y_train):.1%})')
print()
print('Feature statistics after StandardScaler:')
print(f'  mean = {X_train.mean(axis=0).round(4)}')
print(f'  std  = {X_train.std(axis=0).round(4)}')

In [ ]:
# ── EDA ───────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Scatter: standardised training data
for cls_idx in range(2):
    mask = y_train == cls_idx
    axes[0].scatter(X_train[mask, 0], X_train[mask, 1],
                    c=COLORS[cls_idx], alpha=0.55, s=20, label=LABELS[cls_idx])
axes[0].set_title('Training Data (Standardised)', fontsize=12)
axes[0].set_xlabel('Feature 1 (x₁)')
axes[0].set_ylabel('Feature 2 (x₂)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# --- Histogram: feature-1 distribution by class
for cls_idx in range(2):
    mask = y_train == cls_idx
    axes[1].hist(X_train[mask, 0], bins=30, alpha=0.6,
                 color=COLORS[cls_idx], label=LABELS[cls_idx], density=True)
axes[1].set_title('Feature 1 Distribution by Class', fontsize=12)
axes[1].set_xlabel('x₁ (standardised)')
axes[1].set_ylabel('Density')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# --- Noise sensitivity: same moons at three noise levels
noise_levels = [(0.05, 0.9), (0.25, 0.55), (0.45, 0.25)]
for noise_val, alpha_val in noise_levels:
    X_tmp, y_tmp = make_moons(n_samples=300, noise=noise_val, random_state=SEED)
    for cls_idx in range(2):
        mask = y_tmp == cls_idx
        axes[2].scatter(X_tmp[mask, 0], X_tmp[mask, 1],
                        c=COLORS[cls_idx], alpha=alpha_val, s=12)
legend_elems = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='gray',
           markersize=8, alpha=a, label=f'noise={n}')
    for n, a in noise_levels
]
axes[2].legend(handles=legend_elems, fontsize=9)
axes[2].set_title('Sensitivity to Noise Level', fontsize=12)
axes[2].set_xlabel('x₁')
axes[2].set_ylabel('x₂')
axes[2].grid(True, alpha=0.3)

plt.suptitle('EDA — make_moons Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Part 1 — Logistic Regression from Scratch

### 1.1 MLE Derivation of the Sigmoid

We model each label $y_i \in \{0, 1\}$ as a Bernoulli random variable:
$$P(y_i \mid \mathbf{x}_i, \mathbf{w}) = \sigma(\mathbf{w}^\top \mathbf{x}_i)^{y_i} \cdot [1 - \sigma(\mathbf{w}^\top \mathbf{x}_i)]^{1 - y_i}$$

where $\sigma(z) = 1 / (1 + e^{-z})$ is the **sigmoid function**, which maps logits $z = \mathbf{w}^\top \mathbf{x}$ to probabilities in $(0, 1)$.

### 1.2 The Binary Cross-Entropy Loss

Taking the negative log-likelihood and averaging over $n$ samples gives the **binary cross-entropy**:
$$\mathcal{L}(\mathbf{w}) = -\frac{1}{n} \sum_{i=1}^{n} \left[ y_i \log \sigma(z_i) + (1 - y_i) \log(1 - \sigma(z_i)) \right]$$

### 1.3 Gradient Descent Update Rule

Differentiating $\mathcal{L}$ with respect to $\mathbf{w}$ yields a beautifully simple gradient:
$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} \mathbf{X}^\top (\hat{\mathbf{y}} - \mathbf{y})
\quad \text{where} \quad \hat{\mathbf{y}} = \sigma(\mathbf{X}\mathbf{w})$$

The update rule is: $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla_{\mathbf{w}} \mathcal{L}$

In [ ]:
def sigmoid(z: np.ndarray | float) -> np.ndarray | float:
    """Compute the standard logistic (sigmoid) function.

    WARNING: Numerically unstable for large |z| values due to exp overflow.
    Use sigmoid_stable for production use.

    Args:
        z: Input logit — scalar or NumPy array.

    Returns:
        Activation in (0, 1), same shape as z.
    """
    return 1.0 / (1.0 + np.exp(-z))


def sigmoid_stable(z: np.ndarray | float) -> np.ndarray | float:
    """Numerically stable sigmoid via piece-wise evaluation.

    Avoids overflow by splitting into two cases:
        z >= 0:  sigma(z) = 1 / (1 + exp(-z))
        z <  0:  sigma(z) = exp(z) / (1 + exp(z))

    Both branches are bounded and avoid exp overflow / underflow.

    Args:
        z: Input logit — scalar or NumPy array.

    Returns:
        Activation in (0, 1), same shape as z.
    """
    z = np.asarray(z, dtype=float)
    positive = z >= 0
    result = np.where(
        positive,
        1.0 / (1.0 + np.exp(-np.where(positive, z, 0.0))),
        np.exp(np.where(positive, 0.0, z)) / (1.0 + np.exp(np.where(positive, 0.0, z))),
    )
    return result


# ── Stability verification ────────────────────────────────────────────────────
z_test = np.array([-1000.0, -10.0, -1.0, 0.0, 1.0, 10.0, 1000.0])
print('Sigmoid stability test:')
for z_val in z_test:
    stable = float(sigmoid_stable(z_val))
    print(f'  z = {z_val:8.1f}  ->  sigma(z) = {stable:.10f}')
print(f'\nsigma(0)  = {sigmoid_stable(0.0):.6f}  (expected 0.5)')
print(f'sigma(10) = {sigmoid_stable(10.0):.8f}')
print(f'sigma(-10)= {sigmoid_stable(-10.0):.8f}')

In [ ]:
# ── Sigmoid shape and derivative ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

z_range = np.linspace(-7, 7, 500)
sig_vals = sigmoid_stable(z_range)
dsig_vals = sig_vals * (1.0 - sig_vals)   # derivative: sigma * (1 - sigma)

axes[0].plot(z_range, sig_vals, 'b-', linewidth=2.5, label=r'$\sigma(z)$')
axes[0].plot(z_range, dsig_vals, 'g--', linewidth=1.8, alpha=0.85,
             label=r"$\sigma'(z) = \sigma(1-\sigma)$")
axes[0].axhline(y=0.5, color='gray', linestyle=':', alpha=0.7)
axes[0].axvline(x=0,   color='gray', linestyle=':', alpha=0.7)
axes[0].fill_between(z_range, 0.5, sig_vals,
                     where=(z_range >= 0), alpha=0.08, color='blue',
                     label='Class 1 region (z > 0)')
axes[0].fill_between(z_range, sig_vals, 0.5,
                     where=(z_range <= 0), alpha=0.08, color='red',
                     label='Class 0 region (z < 0)')
axes[0].set_xlabel('z (logit)', fontsize=11)
axes[0].set_ylabel(r'$\sigma(z)$', fontsize=11)
axes[0].set_title('Sigmoid Function and Its Derivative', fontsize=12)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(-0.05, 1.05)

# Saturation at extreme z: gradient vanishes
z_extreme = np.array([-7.0, -5.0, -3.0, -1.0, 0.0, 1.0, 3.0, 5.0, 7.0])
grad_extreme = sigmoid_stable(z_extreme) * (1 - sigmoid_stable(z_extreme))
axes[1].bar(range(len(z_extreme)), grad_extreme, color='#7E57C2', alpha=0.8)
axes[1].set_xticks(range(len(z_extreme)))
axes[1].set_xticklabels([f'{z:.0f}' for z in z_extreme])
axes[1].set_xlabel('z value')
axes[1].set_ylabel(r"$\sigma'(z)$")
axes[1].set_title('Gradient Magnitude vs. z\n(near 0 at extremes → vanishing gradient)', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('Key insight: derivative peaks at 0.25 (when z=0) and vanishes at extremes.')
print('This saturation causes the vanishing gradient problem in deep networks.')

### 1.4 Binary Cross-Entropy Loss

The loss for a single sample is:
$$\ell(y_i, \hat{p}_i) = -[y_i \log \hat{p}_i + (1 - y_i) \log(1 - \hat{p}_i)]$$

For $y=1$: loss = $-\log \hat{p}$ (large when $\hat{p}$ is small)
For $y=0$: loss = $-\log(1 - \hat{p})$ (large when $\hat{p}$ is large)

Note: when predictions are random ($\hat{p} = 0.5$), the expected BCE = $\log 2 \approx 0.693$.

In [ ]:
def binary_cross_entropy(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    epsilon: float = 1e-15,
) -> float:
    """Compute mean binary cross-entropy (log-loss).

    Implements:
        L = -(1/n) * sum[ y_i * log(p_i) + (1 - y_i) * log(1 - p_i) ]

    Probabilities are clipped to [epsilon, 1 - epsilon] to prevent log(0).

    Args:
        y_true: Ground-truth binary labels (0 or 1), shape (n_samples,).
        y_prob: Predicted probabilities for class 1, shape (n_samples,).
        epsilon: Numerical clipping value. Default 1e-15.

    Returns:
        Scalar mean log-loss (non-negative float).
    """
    p = np.clip(y_prob, epsilon, 1.0 - epsilon)
    return float(-np.mean(y_true * np.log(p) + (1.0 - y_true) * np.log(1.0 - p)))


# ── Sanity check: loss should decrease as predictions improve ─────────────────
y_toy = np.array([1, 1, 0, 0], dtype=float)
cases = {
    'Perfect   ': np.array([0.999, 0.999, 0.001, 0.001]),
    'Off-by-0.1': np.array([0.6,   0.6,   0.4,   0.4  ]),
    'Random    ': np.array([0.5,   0.5,   0.5,   0.5  ]),
    'Wrong     ': np.array([0.001, 0.001, 0.999, 0.999]),
}
print('Loss sanity check  (expected: perfect < off < random < wrong):')
for name, probs in cases.items():
    print(f'  {name}: BCE = {binary_cross_entropy(y_toy, probs):.6f}')
print(f'\n  log(2) = {np.log(2):.6f}  (theoretical BCE for random guessing)')

### 1.5 L1 Regularisation — Proximal Gradient (Soft Thresholding)

L1 adds $\lambda \sum |w_j|$ to the loss. This penalty is **non-differentiable at $w_j = 0$**, so we cannot simply subtract a gradient. Instead, we use the **proximal gradient** method.

After a standard gradient step, apply the proximal operator element-wise:
$$\text{prox}_{\alpha |\cdot|}(x_j) = \text{sign}(x_j) \cdot \max(|x_j| - \alpha, 0)$$

This **soft-thresholds** each weight: values with $|w_j| \le \alpha$ are set to exactly 0 (sparse!), and larger values are shrunk by $\alpha$. The bias term is excluded from the penalty.

In [ ]:
def soft_threshold(x: np.ndarray, alpha: float) -> np.ndarray:
    """Apply element-wise soft thresholding (proximal operator for L1 norm).

    Computes:
        prox_{alpha * |.|} (x_i) = sign(x_i) * max(|x_i| - alpha, 0)

    Values inside [-alpha, alpha] are mapped to exactly 0 (induces sparsity).
    Values outside are shrunk toward zero by alpha.

    Args:
        x: Weight values to threshold, shape (n_features,).
        alpha: Threshold magnitude (= learning_rate * lambda).

    Returns:
        Thresholded array, same shape as x.
    """
    return np.sign(x) * np.maximum(np.abs(x) - alpha, 0.0)


# ── Visualise the operator ────────────────────────────────────────────────────
x_demo  = np.linspace(-3.5, 3.5, 500)
alphas  = [0.5, 1.0, 1.5]

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(x_demo, x_demo, 'k--', alpha=0.35, linewidth=1.2, label='identity (no reg)')
for alpha_val in alphas:
    y_demo = soft_threshold(x_demo, alpha=alpha_val)
    ax.plot(x_demo, y_demo, linewidth=2.0, label=f'\u03b1 = {alpha_val}')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.set_xlabel('x (weight value)', fontsize=11)
ax.set_ylabel('prox(x)', fontsize=11)
ax.set_title('Soft-Threshold Operator \u2014 Proximal Step for L1', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Quick numerical demo
x_vals = np.array([-3.0, -1.5, -0.5, 0.0, 0.5, 1.5, 3.0])
print('Soft-threshold demo  (\u03b1 = 1.0):')
print(f'  input  : {x_vals}')
print(f'  output : {soft_threshold(x_vals, 1.0)}')
print('  Values inside [-1, 1] collapse to 0.0  \u2192  sparsity')

### 1.6 Gradient of the Binary Cross-Entropy

For the BCE loss with **L2 regularisation** ($\lambda = 1 / (C \cdot n)$):
$$\nabla_{\mathbf{w}} \mathcal{L} = \frac{1}{n} \mathbf{X}^\top (\hat{\mathbf{y}} - \mathbf{y}) + \lambda \tilde{\mathbf{w}}$$

where $\tilde{\mathbf{w}}$ is $\mathbf{w}$ with the bias weight zeroed (we do not regularise the bias). For **L1**, only the unregularised BCE gradient is used here; the $\ell_1$ penalty is handled by the proximal step after the gradient update.

In [ ]:
def compute_gradient(
    X_design: np.ndarray,
    y: np.ndarray,
    weights: np.ndarray,
    penalty: str = 'none',
    C: float = 1.0,
) -> tuple[float, np.ndarray]:
    """Compute BCE loss and gradient w.r.t. the weight vector.

    Gradient (no reg):  dL/dw = (1/n) * X^T * (sigma(Xw) - y)
    Gradient (L2):      dL/dw += lambda * [0, w_1, ..., w_p]  (bias excluded)
    Gradient (L1):      same as no-reg; proximal step applied externally.

    Args:
        X_design: Design matrix with prepended bias column, shape (n, p+1).
        y: Binary labels (0 or 1), shape (n,).
        weights: Current weight vector, shape (p+1,).
        penalty: Regularisation type: 'none', 'l2', or 'l1'. Default 'none'.
        C: Inverse regularisation strength (larger = weaker). Default 1.0.

    Returns:
        Tuple of (scalar_loss, gradient_array). Gradient has shape (p+1,).
    """
    n = len(y)
    eps = 1e-15
    y_prob    = sigmoid_stable(X_design @ weights)
    y_clipped = np.clip(y_prob, eps, 1.0 - eps)
    bce = float(-np.mean(y * np.log(y_clipped) + (1.0 - y) * np.log(1.0 - y_clipped)))

    error = y_prob - y
    grad  = (X_design.T @ error) / n

    lambda_val = 1.0 / (C * n)
    if penalty == 'l2':
        reg_weights    = np.concatenate([[0.0], weights[1:]])   # skip bias
        grad           = grad + lambda_val * reg_weights
        bce            = bce + 0.5 * lambda_val * float(np.sum(weights[1:] ** 2))
    elif penalty == 'l1':
        bce = bce + lambda_val * float(np.sum(np.abs(weights[1:])))

    return bce, grad

In [ ]:
def check_gradient_numerically(
    X_design: np.ndarray,
    y: np.ndarray,
    weights: np.ndarray,
    epsilon: float = 1e-5,
) -> dict[str, object]:
    """Verify analytic gradient against centered finite differences.

    Numerical gradient for each weight w_i:
        dL/dw_i ~= [L(w + eps*e_i) - L(w - eps*e_i)] / (2 * eps)

    A relative error below 1e-5 confirms the gradient implementation is correct.

    Args:
        X_design: Design matrix with bias, shape (n, p+1).
        y: Binary labels (0 or 1), shape (n,).
        weights: Weight vector to evaluate at, shape (p+1,).
        epsilon: Finite-difference step size. Default 1e-5.

    Returns:
        Dict with keys 'analytic', 'numerical', 'max_error', 'relative_error'.
    """
    _, analytic_grad = compute_gradient(X_design, y, weights)
    numerical_grad = np.zeros_like(weights)

    for i in range(len(weights)):
        w_pos    = weights.copy()
        w_pos[i] += epsilon
        loss_pos, _ = compute_gradient(X_design, y, w_pos)

        w_neg    = weights.copy()
        w_neg[i] -= epsilon
        loss_neg, _ = compute_gradient(X_design, y, w_neg)

        numerical_grad[i] = (loss_pos - loss_neg) / (2.0 * epsilon)

    abs_diff       = np.abs(analytic_grad - numerical_grad)
    max_error      = float(np.max(abs_diff))
    denom          = np.linalg.norm(analytic_grad) + np.linalg.norm(numerical_grad) + 1e-10
    relative_error = float(np.linalg.norm(analytic_grad - numerical_grad) / denom)

    return {
        'analytic':       analytic_grad,
        'numerical':      numerical_grad,
        'max_error':      max_error,
        'relative_error': relative_error,
    }


# ── Run gradient check ────────────────────────────────────────────────────────
n_check  = 60
X_chk    = np.column_stack([np.ones(n_check), X_train[:n_check]])
w_chk    = rng.normal(0.0, 0.1, X_chk.shape[1])

result = check_gradient_numerically(X_chk, y_train[:n_check], w_chk)
print('Gradient check — BCE (no regularisation):')
print(f'  Max absolute error  : {result["max_error"]:.2e}')
print(f'  Relative error      : {result["relative_error"]:.2e}')
print(f'  Analytic  (first 3) : {result["analytic"][:3].round(6)}')
print(f'  Numerical (first 3) : {result["numerical"][:3].round(6)}')
threshold = 1e-5
status = 'PASS' if result['max_error'] < threshold else 'FAIL'
print(f'\n  [{status}] max_error {result["max_error"]:.2e} < threshold {threshold:.0e}')

### 1.7 Gradient Descent Training Loop

With the sigmoid, loss, and gradient in hand, the training loop is straightforward:

1. Compute $\hat{\mathbf{y}} = \sigma(\mathbf{X}\mathbf{w})$
2. Compute loss and gradient: $g = \frac{1}{n}\mathbf{X}^\top(\hat{\mathbf{y}} - \mathbf{y}) + \lambda \tilde{\mathbf{w}}$
3. Update: $\mathbf{w} \leftarrow \mathbf{w} - \eta g$
4. For L1: apply proximal step $\mathbf{w}[1:] \leftarrow \text{prox}_{\eta\lambda}(\mathbf{w}[1:])$
5. Repeat until convergence ($|\Delta\mathcal{L}| < \epsilon$)

In [ ]:
def run_gradient_descent(
    X_design: np.ndarray,
    y: np.ndarray,
    learning_rate: float = LEARNING_RATE,
    num_epochs: int = NUM_EPOCHS,
    tolerance: float = TOLERANCE,
    penalty: str = 'none',
    C: float = C_DEFAULT,
    random_state: int = SEED,
    verbose: bool = True,
) -> tuple[np.ndarray, list[float]]:
    """Standalone gradient descent loop for logistic regression.

    Demonstrates the raw update mechanics before wrapping them into the
    LogisticRegressionScratch class in Part 2. Supports 'none', 'l2',
    and 'l1' regularisation. L1 uses the proximal gradient method.

    Args:
        X_design: Design matrix with bias column, shape (n, p+1).
        y: Binary labels (0 or 1), shape (n,).
        learning_rate: Step size. Default LEARNING_RATE.
        num_epochs: Maximum iterations. Default NUM_EPOCHS.
        tolerance: Early-stopping threshold |delta_loss|. Default TOLERANCE.
        penalty: Regularisation type: 'none', 'l2', 'l1'. Default 'none'.
        C: Inverse regularisation strength. Default C_DEFAULT.
        random_state: Seed for weight initialisation. Default SEED.
        verbose: Print convergence messages. Default True.

    Returns:
        Tuple of (final_weights, loss_history) where loss_history is a list
        of per-epoch loss values.
    """
    init_rng = np.random.default_rng(seed=random_state)
    weights  = init_rng.normal(0.0, 0.01, X_design.shape[1])
    loss_history: list[float] = []
    n          = X_design.shape[0]
    lambda_val = 1.0 / (C * n)
    prev_loss  = np.inf

    for epoch in range(num_epochs):
        loss, grad = compute_gradient(X_design, y, weights, penalty=penalty, C=C)
        loss_history.append(loss)

        weights = weights - learning_rate * grad

        if penalty == 'l1':
            step          = learning_rate * lambda_val
            weights[1:] = soft_threshold(weights[1:], step)

        if abs(prev_loss - loss) < tolerance:
            if verbose:
                print(f'  Converged at epoch {epoch + 1:,} / {num_epochs:,}'
                      f'  |\u0394loss| = {abs(prev_loss - loss):.2e}')
            break
        prev_loss = loss

    return weights, loss_history


# ── Demo: train without regularisation ───────────────────────────────────────
n_tr           = X_train.shape[0]
X_design_train = np.column_stack([np.ones(n_tr), X_train])
X_design_test  = np.column_stack([np.ones(X_test.shape[0]), X_test])

print('Running gradient descent (no regularisation) ...')
w_gd, losses_gd = run_gradient_descent(X_design_train, y_train)

y_prob_gd   = sigmoid_stable(X_design_test @ w_gd)
y_pred_gd   = (y_prob_gd >= 0.5).astype(int)
gd_test_acc = float(np.mean(y_pred_gd == y_test))
print(f'\n  Final loss    : {losses_gd[-1]:.6f}')
print(f'  Test accuracy : {gd_test_acc:.4f}')
print(f'  Weights (bias, w1, w2) : {w_gd.round(4)}')

In [ ]:
# ── Loss curve: linear and log-convergence view ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

epoch_axis = np.arange(1, len(losses_gd) + 1)

axes[0].plot(epoch_axis, losses_gd, 'b-', linewidth=1.5)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Binary Cross-Entropy Loss')
axes[0].set_title('Gradient Descent Convergence', fontsize=12)
axes[0].grid(True, alpha=0.3)

min_loss = losses_gd[-1]
log_diff = np.log10(np.maximum(np.array(losses_gd) - min_loss, 1e-12))
axes[1].plot(epoch_axis, log_diff, 'r-', linewidth=1.5)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('log\u2081\u2080(L \u2212 L*)')
axes[1].set_title('Convergence Rate (Log Scale)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Training summary:')
print(f'  Initial loss   : {losses_gd[0]:.6f}')
print(f'  Final loss     : {losses_gd[-1]:.6f}')
print(f'  Total epochs   : {len(losses_gd):,}')
print(f'  Loss reduction : {(losses_gd[0] - losses_gd[-1]) / losses_gd[0]:.2%}')

---
## Part 2 — Putting It All Together

We now combine the building blocks into `LogisticRegressionScratch` — a scikit-learn–compatible class with `fit` / `predict` / `predict_proba` / `score` methods.

### L1 vs L2 Regularisation — Geometric Intuition

Both penalties constrain the weight vector to a budget ball:
- **L2** ($\|\mathbf{w}\|_2 \le r$): The ball is a **sphere** (circle in 2D). Loss contours touch the constraint surface at smooth points → weights **shrink but stay non-zero**.
- **L1** ($\|\mathbf{w}\|_1 \le r$): The ball is a **diamond** (cross-polytope). Loss contours tend to first touch the constraint at **corners** (one weight = 0) → **sparse solutions**.

The regularisation parameter $\lambda = 1 / (C \cdot n)$; a large $C$ means weak regularisation.

In [ ]:
class LogisticRegressionScratch:
    """Logistic regression with optional L1 or L2 regularisation.

    Trains via full-batch gradient descent. L2 adds a penalty term to the
    gradient; L1 uses proximal gradient descent (soft thresholding) so that
    weights are exactly zeroed when they fall inside the threshold radius.
    The bias term is always excluded from the regularisation penalty.

    Args:
        penalty: Regularisation type. One of 'none', 'l2', 'l1'. Default 'none'.
        C: Inverse regularisation strength. Larger C = weaker regularisation.
           Must be positive. Default 1.0.
        learning_rate: Gradient descent step size. Default 0.1.
        num_epochs: Maximum gradient descent iterations. Default 2000.
        tolerance: Stop early when |delta_loss| < tolerance. Default 1e-7.
        random_state: Seed for weight initialisation. Default None.

    Attributes:
        train_losses_ (list[float]): Per-epoch loss values (populated after fit).
        is_fitted_ (bool): Whether fit() has been called.
    """

    def __init__(
        self,
        penalty: str = 'none',
        C: float = 1.0,
        learning_rate: float = 0.1,
        num_epochs: int = 2000,
        tolerance: float = 1e-7,
        random_state: int | None = None,
    ) -> None:
        """Initialise logistic regression with specified hyperparameters.

        Args:
            penalty: Regularisation type. One of 'none', 'l2', 'l1'. Default 'none'.
            C: Inverse regularisation strength. Must be positive. Default 1.0.
            learning_rate: Gradient descent step size. Default 0.1.
            num_epochs: Maximum gradient descent iterations. Default 2000.
            tolerance: Stop early when |delta_loss| < tolerance. Default 1e-7.
            random_state: Seed for weight initialisation. Default None.
        """
        if penalty not in ('none', 'l1', 'l2'):
            raise ValueError(f"penalty must be 'none', 'l1', or 'l2', got {penalty!r}")
        if C <= 0:
            raise ValueError(f'C must be positive, got {C}')
        self.penalty       = penalty
        self.C             = C
        self.learning_rate = learning_rate
        self.num_epochs    = num_epochs
        self.tolerance     = tolerance
        self.random_state  = random_state
        self._weights: np.ndarray | None = None
        self.train_losses_: list[float]  = []
        self.is_fitted_: bool            = False

    def _add_bias(self, X: np.ndarray) -> np.ndarray:
        """Prepend a column of ones to X for the bias term.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Design matrix, shape (n_samples, n_features + 1).
        """
        return np.column_stack([np.ones(X.shape[0]), X])

    def _init_weights(self, n_features: int) -> np.ndarray:
        """Initialise weight vector with small random values.

        Args:
            n_features: Number of parameters (features + 1 bias).

        Returns:
            Weight vector of shape (n_features,).
        """
        init_rng = np.random.default_rng(seed=self.random_state)
        return init_rng.normal(loc=0.0, scale=0.01, size=n_features)

    def fit(self, X: np.ndarray, y: np.ndarray) -> 'LogisticRegressionScratch':
        """Fit logistic regression via gradient descent (+ proximal step for L1).

        Update rule (each epoch):
            1. g  = (1/n) * X^T * (sigma(Xw) - y)  +  lambda * [0, w_1, ..., w_p]   (L2)
            2. w  = w - lr * g
            3. w[1:] = soft_threshold(w[1:], lr * lambda)                              (L1 only)

        Args:
            X: Training features, shape (n_samples, n_features).
            y: Binary labels (0 or 1), shape (n_samples,).

        Returns:
            self (allows method chaining: model.fit(X, y).predict(X)).
        """
        X_d = self._add_bias(X)
        n, p = X_d.shape
        self._weights      = self._init_weights(p)
        self.train_losses_ = []
        lambda_val         = 1.0 / (self.C * n)
        prev_loss          = np.inf

        for epoch in range(self.num_epochs):
            y_prob    = sigmoid_stable(X_d @ self._weights)
            eps       = 1e-15
            y_clipped = np.clip(y_prob, eps, 1.0 - eps)
            bce       = -np.mean(y * np.log(y_clipped) + (1.0 - y) * np.log(1.0 - y_clipped))

            if self.penalty == 'l2':
                reg = 0.5 * lambda_val * float(np.sum(self._weights[1:] ** 2))
            elif self.penalty == 'l1':
                reg = lambda_val * float(np.sum(np.abs(self._weights[1:])))
            else:
                reg = 0.0
            loss = float(bce) + reg
            self.train_losses_.append(loss)

            error = y_prob - y
            grad  = (X_d.T @ error) / n
            if self.penalty == 'l2':
                grad[1:] = grad[1:] + lambda_val * self._weights[1:]

            self._weights = self._weights - self.learning_rate * grad

            if self.penalty == 'l1':
                step              = self.learning_rate * lambda_val
                self._weights[1:] = soft_threshold(self._weights[1:], step)

            if abs(prev_loss - loss) < self.tolerance:
                break
            prev_loss = loss

        self.is_fitted_ = True
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities.

        Args:
            X: Feature matrix, shape (n_samples, n_features).

        Returns:
            Array of shape (n_samples, 2) with columns [P(y=0), P(y=1)].
        """
        if not self.is_fitted_:
            raise RuntimeError('Call fit() before predict_proba().')
        p1 = sigmoid_stable(self._add_bias(X) @ self._weights)
        return np.column_stack([1.0 - p1, p1])

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Predict binary class labels.

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            threshold: Decision threshold for class 1. Default 0.5.

        Returns:
            Binary predictions, shape (n_samples,), dtype int.
        """
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

    def score(self, X: np.ndarray, y: np.ndarray) -> float:
        """Compute classification accuracy on (X, y).

        Args:
            X: Feature matrix, shape (n_samples, n_features).
            y: True labels, shape (n_samples,).

        Returns:
            Fraction of correctly classified samples in [0, 1].
        """
        return float(np.mean(self.predict(X) == y))

    @property
    def coef_(self) -> np.ndarray:
        """Feature weights (excludes bias), shape (n_features,)."""
        if self._weights is None:
            raise RuntimeError('Call fit() first.')
        return self._weights[1:]

    @property
    def intercept_(self) -> float:
        """Bias (intercept) term as a scalar."""
        if self._weights is None:
            raise RuntimeError('Call fit() first.')
        return float(self._weights[0])

In [ ]:
# ── Sanity check: linearly separable blobs ────────────────────────────────────
X_sep, y_sep = make_blobs(
    n_samples=200, centers=[[-2.5, -2.5], [2.5, 2.5]],
    cluster_std=0.8, random_state=SEED
)
scaler_sep = StandardScaler()
X_sep_sc   = scaler_sep.fit_transform(X_sep)

X_sep_tr, X_sep_te, y_sep_tr, y_sep_te = train_test_split(
    X_sep_sc, y_sep, test_size=0.25, random_state=SEED
)

model_sanity = LogisticRegressionScratch(
    penalty='none', learning_rate=0.1, num_epochs=1000,
    tolerance=1e-8, random_state=SEED
)
model_sanity.fit(X_sep_tr, y_sep_tr)

train_acc = model_sanity.score(X_sep_tr, y_sep_tr)
test_acc  = model_sanity.score(X_sep_te, y_sep_te)
print('Sanity check — linearly separable blobs:')
print(f'  Train accuracy : {train_acc:.4f}')
print(f'  Test  accuracy : {test_acc:.4f}  (expected close to 1.00)')
print(f'  Coef           : {model_sanity.coef_.round(4)}')
print(f'  Intercept      : {model_sanity.intercept_:.4f}')
assert train_acc > 0.95, f'Expected > 0.95 on separable data, got {train_acc:.4f}'
print('\n  [PASS] LogisticRegressionScratch works on linearly separable data.')

---
## Part 3 — Training on make_moons

We train three variants — no regularisation, L2 (Ridge), and L1 (Lasso) — on the same `make_moons` dataset, compare their decision boundaries, visualise the geometric constraint regions that explain *why* L1 produces sparsity, and validate against `sklearn.LogisticRegression`.

In [ ]:
# ── Train three regularisation variants ──────────────────────────────────────
configs = [
    ('No Regularisation', 'none', C_DEFAULT),
    ('L2 Regularisation', 'l2',   C_DEFAULT),
    ('L1 Regularisation', 'l1',   C_DEFAULT),
]

models: dict[str, LogisticRegressionScratch] = {}

for name, penalty, C in configs:
    print(f'Training : {name}  (penalty={penalty!r}, C={C}) ...')
    model = LogisticRegressionScratch(
        penalty=penalty,
        C=C,
        learning_rate=LEARNING_RATE,
        num_epochs=NUM_EPOCHS,
        tolerance=TOLERANCE,
        random_state=SEED,
    )
    model.fit(X_train, y_train)
    train_acc = model.score(X_train, y_train)
    test_acc  = model.score(X_test,  y_test)
    models[name] = model
    print(f'  Train acc : {train_acc:.4f}  |  Test acc : {test_acc:.4f}')
    print(f'  coef      : {model.coef_.round(4)}')
    print(f'  intercept : {model.intercept_:.4f}')
    print()

In [ ]:
def plot_decision_boundary(
    model: LogisticRegressionScratch,
    X: np.ndarray,
    y: np.ndarray,
    title: str,
    ax: plt.Axes,
    resolution: float = 0.02,
) -> None:
    """Render the decision boundary of a fitted logistic regression model.

    Evaluates the model on a dense grid over the feature space and overlays
    the training samples with colour indicating their true class.

    Args:
        model: Fitted LogisticRegressionScratch instance.
        X: Feature matrix (must have exactly 2 columns), shape (n, 2).
        y: Binary labels, shape (n,).
        title: Plot title string.
        ax: Matplotlib Axes object to draw on.
        resolution: Grid step size. Smaller = crisper boundary. Default 0.02.

    Returns:
        None (modifies ax in place).
    """
    x1_min, x1_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x2_min, x2_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx1, xx2 = np.meshgrid(
        np.arange(x1_min, x1_max, resolution),
        np.arange(x2_min, x2_max, resolution),
    )
    grid_points = np.column_stack([xx1.ravel(), xx2.ravel()])
    probs       = model.predict_proba(grid_points)[:, 1].reshape(xx1.shape)

    ax.contourf(xx1, xx2, probs, levels=50, cmap='RdBu_r', alpha=0.3, vmin=0, vmax=1)
    ax.contour(xx1, xx2, probs, levels=[0.5], colors='black', linewidths=1.5)

    for cls_idx in range(2):
        mask = y == cls_idx
        ax.scatter(X[mask, 0], X[mask, 1],
                   c=COLORS[cls_idx], s=15, alpha=0.6, edgecolors='none')

    acc = model.score(X_test, y_test)
    ax.set_title(f'{title}\n(test acc: {acc:.3f})', fontsize=11)
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)
    ax.grid(True, alpha=0.2)


# ── Side-by-side boundary plots ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, _, _) in zip(axes, configs):
    plot_decision_boundary(models[name], X_train, y_train, title=name, ax=ax)
plt.suptitle('Decision Boundaries — make_moons', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.1 Geometric Interpretation: Why L1 Induces Sparsity

Regularisation constrains the solution to a **budget ball** in weight space. The loss contours (ellipses) are pushed outward from the unregularised optimum. The regularised solution is where the contours **first touch** the constraint:

- **L2 ball (sphere):** Smooth surface → contours touch at a **smooth point** → weights shrink but remain non-zero.
- **L1 ball (diamond):** Has **corners at the axes** → contours typically touch at a corner where one or more weights equal 0 → sparsity.

In [ ]:
# ── Constraint region visualisation ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
theta  = np.linspace(0, 2 * np.pi, 400)
radius = 1.5

# Simulated loss contours centred at the (unregularised) optimum
optimum = np.array([1.9, 0.6])
contour_sizes = [0.4, 0.8, 1.3, 1.9, 2.6]

# --- L2 constraint: ||w||_2 <= r  (circle)
axes[0].fill(radius * np.cos(theta), radius * np.sin(theta),
             alpha=0.12, color='blue')
axes[0].plot(radius * np.cos(theta), radius * np.sin(theta),
             'b-', linewidth=2.2, label=f'L2 ball: ||w||₂ \u2264 {radius}')
for csize in contour_sizes:
    xc = csize * 1.2 * np.cos(theta) + optimum[0]
    yc = csize * 0.7 * np.sin(theta) + optimum[1]
    axes[0].plot(xc, yc, 'gray', linewidth=0.8, alpha=0.55)
l2_solution = np.array([0.85, 1.25])
axes[0].plot(*l2_solution, 'bs', markersize=11,
             label='L2 solution (smooth, non-sparse)')
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].axvline(0, color='k', linewidth=0.5)
axes[0].set_aspect('equal')
axes[0].legend(fontsize=9, loc='lower right')
axes[0].set_title('L2 Regularisation\n(Circle \u2192 No Sparsity)', fontsize=12)
axes[0].set_xlabel('w₁')
axes[0].set_ylabel('w₂')
axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)

# --- L1 constraint: |w_1| + |w_2| <= r  (diamond)
diamond_x = [radius, 0, -radius, 0, radius]
diamond_y = [0, radius, 0, -radius, 0]
axes[1].fill(diamond_x, diamond_y, alpha=0.12, color='red')
axes[1].plot(diamond_x, diamond_y, 'r-', linewidth=2.2,
             label=f'L1 ball: ||w||₁ \u2264 {radius}')
for csize in contour_sizes:
    xc = csize * 1.2 * np.cos(theta) + optimum[0]
    yc = csize * 0.7 * np.sin(theta) + optimum[1]
    axes[1].plot(xc, yc, 'gray', linewidth=0.8, alpha=0.55)
l1_solution = np.array([radius, 0.0])
axes[1].plot(*l1_solution, 'r^', markersize=12,
             label='L1 solution (corner \u2192 w₂ = 0!)')
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].axvline(0, color='k', linewidth=0.5)
axes[1].set_aspect('equal')
axes[1].legend(fontsize=9, loc='lower right')
axes[1].set_title('L1 Regularisation\n(Diamond \u2192 Sparsity at Corners)', fontsize=12)
axes[1].set_xlabel('w₁')
axes[1].set_ylabel('w₂')
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)

plt.suptitle('Why L1 Induces Sparsity but L2 Does Not', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
print('Geometric insight:')
print('  L2 (circle)  : contours touch smooth surface \u2192 weights shrink, stay non-zero')
print('  L1 (diamond) : contours touch corner \u2192 one weight snaps to exactly 0')

In [ ]:
# ── Coefficient comparison bar chart ─────────────────────────────────────────
model_noreg = models['No Regularisation']
model_l2    = models['L2 Regularisation']
model_l1    = models['L1 Regularisation']

feature_names = ['x₁', 'x₂']
n_features    = len(feature_names)
x_pos         = np.arange(n_features)
width         = 0.26

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_pos - width, model_noreg.coef_, width, label='No Reg',  color='#9E9E9E', alpha=0.88)
ax.bar(x_pos,         model_l2.coef_,    width, label='L2 (Ridge)', color='#2196F3', alpha=0.88)
ax.bar(x_pos + width, model_l1.coef_,    width, label='L1 (Lasso)', color='#F44336', alpha=0.88)
ax.set_xticks(x_pos)
ax.set_xticklabels(feature_names, fontsize=12)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Weight value')
ax.set_title('Learned Coefficients: No Reg vs L2 vs L1', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print('Coefficient summary:')
for label, m in [('No Reg', model_noreg), ('L2', model_l2), ('L1', model_l1)]:
    nonzero = int(np.sum(np.abs(m.coef_) > 1e-6))
    print(f'  {label:6s}: coef={m.coef_.round(4)},'
          f' intercept={m.intercept_:.4f},'
          f' nonzero={nonzero}/{len(m.coef_)}')

In [ ]:
def compute_reg_path(
    X_tr: np.ndarray,
    y_tr: np.ndarray,
    X_te: np.ndarray,
    y_te: np.ndarray,
    c_values: list[float],
    penalty: str,
    num_epochs: int = 1500,
) -> tuple[np.ndarray, list[float], list[float]]:
    """Train models over a range of C values and record coefficients and accuracy.

    Args:
        X_tr: Training features, shape (n_train, n_features).
        y_tr: Training labels, shape (n_train,).
        X_te: Test features, shape (n_test, n_features).
        y_te: Test labels, shape (n_test,).
        c_values: List of C (inverse regularisation strength) values.
        penalty: Regularisation type: 'l1' or 'l2'.
        num_epochs: Max iterations per model. Default 1500.

    Returns:
        Tuple (coef_matrix, train_accs, test_accs). coef_matrix has shape
        (len(c_values), n_features).
    """
    coef_list:   list[np.ndarray] = []
    train_accs:  list[float]      = []
    test_accs:   list[float]      = []

    for C_val in c_values:
        model = LogisticRegressionScratch(
            penalty=penalty, C=C_val,
            learning_rate=LEARNING_RATE, num_epochs=num_epochs,
            tolerance=TOLERANCE, random_state=SEED,
        )
        model.fit(X_tr, y_tr)
        coef_list.append(model.coef_.copy())
        train_accs.append(model.score(X_tr, y_tr))
        test_accs.append(model.score(X_te, y_te))

    return np.array(coef_list), train_accs, test_accs


print('Computing regularisation paths ...')
coefs_l2, tr_accs_l2, te_accs_l2 = compute_reg_path(
    X_train, y_train, X_test, y_test, C_VALUES, penalty='l2'
)
coefs_l1, tr_accs_l1, te_accs_l1 = compute_reg_path(
    X_train, y_train, X_test, y_test, C_VALUES, penalty='l1'
)

feat_colors = ['#E53935', '#1E88E5']
feat_labels = ['w₁ (x₁)', 'w₂ (x₂)']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for coefs, title, ax in [
    (coefs_l2, 'L2 Coefficient Path', axes[0]),
    (coefs_l1, 'L1 Coefficient Path', axes[1]),
]:
    for feat_idx in range(coefs.shape[1]):
        ax.semilogx(C_VALUES, coefs[:, feat_idx], 'o-',
                    color=feat_colors[feat_idx], label=feat_labels[feat_idx])
    ax.axhline(0, color='gray', linewidth=0.7)
    ax.set_xlabel('C (inverse regularisation strength)')
    ax.set_ylabel('Coefficient value')
    ax.set_title(title, fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[2].semilogx(C_VALUES, te_accs_l2, 'b-o', label='L2 test',  linewidth=2)
axes[2].semilogx(C_VALUES, te_accs_l1, 'r--s', label='L1 test', linewidth=2)
axes[2].set_xlabel('C (inverse regularisation strength)')
axes[2].set_ylabel('Accuracy')
axes[2].set_title('Test Accuracy vs. Regularisation Strength', fontsize=12)
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### 3.2 Library Comparison

We verify our from-scratch implementation against `sklearn.linear_model.LogisticRegression`. Coefficients should be numerically close for L2; small differences are expected for L1 (sklearn uses coordinate descent, we use proximal gradient).

In [ ]:
# ── sklearn comparison ────────────────────────────────────────────────────────
sklearn_configs = [
    ('sklearn — No Reg', SklearnLR(penalty=None, solver='lbfgs',
                                    max_iter=5000, random_state=SEED)),
    ('sklearn — L2',     SklearnLR(penalty='l2', C=C_DEFAULT, solver='lbfgs',
                                    max_iter=5000, random_state=SEED)),
    ('sklearn — L1',     SklearnLR(penalty='l1', C=C_DEFAULT, solver='liblinear',
                                    max_iter=5000, random_state=SEED)),
]

print(f'{"Model":<28} {"Train":>8} {"Test":>8} {"[w1, w2]":>20} {"Intercept":>12}')
print('-' * 80)

for name, sk_model in sklearn_configs:
    sk_model.fit(X_train, y_train)
    tr_acc   = sk_model.score(X_train, y_train)
    te_acc   = sk_model.score(X_test,  y_test)
    coef_str = f'[{sk_model.coef_[0, 0]:+.4f}, {sk_model.coef_[0, 1]:+.4f}]'
    print(f'  {name:<26} {tr_acc:>8.4f} {te_acc:>8.4f} {coef_str:>20} {sk_model.intercept_[0]:>12.4f}')

print('\nFrom-scratch:')
for name, _, _ in configs:
    m        = models[name]
    coef_str = f'[{m.coef_[0]:+.4f}, {m.coef_[1]:+.4f}]'
    print(f'  {name:<26} {m.score(X_train, y_train):>8.4f} {m.score(X_test, y_test):>8.4f}'
          f' {coef_str:>20} {m.intercept_:>12.4f}')

---
## Part 4 — Evaluation & Analysis

We now build a full evaluation suite from scratch: classification metrics, confusion matrix, ROC curve and AUC. We then run an ablation on regularisation strength and examine the failure cases.

In [ ]:
def compute_classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    y_prob: np.ndarray,
    model_name: str = '',
) -> dict[str, float]:
    """Compute standard binary classification metrics from scratch.

    Metrics:
        accuracy  = (TP + TN) / n
        precision = TP / (TP + FP)
        recall    = TP / (TP + FN)
        f1        = 2 * precision * recall / (precision + recall)
        log_loss  = binary cross-entropy(y_true, y_prob)

    Args:
        y_true: Ground-truth labels (0 or 1), shape (n,).
        y_pred: Predicted labels (0 or 1), shape (n,).
        y_prob: Predicted probability for class 1, shape (n,).
        model_name: Optional label for console output. Default ''.

    Returns:
        Dict mapping metric names to float values.
    """
    tp = float(np.sum((y_pred == 1) & (y_true == 1)))
    tn = float(np.sum((y_pred == 0) & (y_true == 0)))
    fp = float(np.sum((y_pred == 1) & (y_true == 0)))
    fn = float(np.sum((y_pred == 0) & (y_true == 1)))
    n  = float(len(y_true))

    accuracy  = (tp + tn) / n
    precision = tp / (tp + fp + 1e-10)
    recall    = tp / (tp + fn + 1e-10)
    f1        = 2.0 * precision * recall / (precision + recall + 1e-10)
    logloss   = binary_cross_entropy(y_true, y_prob)

    metrics: dict[str, float] = {
        'accuracy':  round(accuracy,  6),
        'precision': round(precision, 6),
        'recall':    round(recall,    6),
        'f1':        round(f1,        6),
        'log_loss':  round(logloss,   6),
    }

    if model_name:
        print(f'{model_name}:')
        for k, v in metrics.items():
            print(f'  {k:<12}: {v:.4f}')
        print()

    return metrics


# ── Compute metrics for all three models ─────────────────────────────────────
all_metrics: dict[str, dict[str, float]] = {}
for name, _, _ in configs:
    y_pred = models[name].predict(X_test)
    y_prob = models[name].predict_proba(X_test)[:, 1]
    all_metrics[name] = compute_classification_metrics(
        y_true=y_test, y_pred=y_pred, y_prob=y_prob, model_name=name
    )

In [ ]:
def confusion_matrix_scratch(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    n_classes: int = 2,
) -> np.ndarray:
    """Compute confusion matrix without using sklearn.

    Entry [i, j] = number of samples with true label i predicted as label j.

    Args:
        y_true: True labels, shape (n_samples,).
        y_pred: Predicted labels, shape (n_samples,).
        n_classes: Number of classes. Default 2.

    Returns:
        Integer confusion matrix of shape (n_classes, n_classes).
    """
    cm = np.zeros((n_classes, n_classes), dtype=int)
    for true_label, pred_label in zip(y_true, y_pred):
        cm[int(true_label), int(pred_label)] += 1
    return cm


# ── Plot confusion matrices for all three models ──────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, _, _) in zip(axes, configs):
    y_pred = models[name].predict(X_test)
    cm     = confusion_matrix_scratch(y_test, y_pred)
    ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_title(f'{name}\n(Confusion Matrix)', fontsize=10)
    ax.set_xlabel('Predicted label')
    ax.set_ylabel('True label')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    cm_max = max(cm.max(), 1)
    for row in range(2):
        for col in range(2):
            color = 'white' if cm[row, col] > cm_max * 0.6 else 'black'
            ax.text(col, row, str(cm[row, col]),
                    ha='center', va='center', color=color,
                    fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# Verify against sklearn
y_pred_nr = models['No Regularisation'].predict(X_test)
cm_scratch = confusion_matrix_scratch(y_test, y_pred_nr)
cm_sklearn  = sk_confusion_matrix(y_test, y_pred_nr)
print(f'Confusion matrix matches sklearn: {np.array_equal(cm_scratch, cm_sklearn)}')

In [ ]:
def roc_curve_scratch(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    n_thresholds: int = 250,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Compute ROC curve by sweeping decision thresholds from 0 to 1.

    At each threshold t, classifies as positive when y_prob >= t, then
    records the true-positive rate (recall) and false-positive rate.

    Args:
        y_true: True binary labels (0 or 1), shape (n_samples,).
        y_prob: Predicted probabilities for class 1, shape (n_samples,).
        n_thresholds: Number of threshold values to sweep. Default 250.

    Returns:
        Tuple of (fpr_array, tpr_array, threshold_array).
    """
    thresholds = np.linspace(0.0, 1.0, n_thresholds)
    thresholds = np.concatenate([[1.0 + 1e-8], thresholds[::-1], [-1e-8]])

    pos = float(np.sum(y_true == 1))
    neg = float(np.sum(y_true == 0))
    fprs: list[float] = []
    tprs: list[float] = []

    for thresh in thresholds:
        y_pred_t = (y_prob >= thresh).astype(int)
        tp = float(np.sum((y_pred_t == 1) & (y_true == 1)))
        fp = float(np.sum((y_pred_t == 1) & (y_true == 0)))
        tprs.append(tp / (pos + 1e-10))
        fprs.append(fp / (neg + 1e-10))

    return np.array(fprs), np.array(tprs), thresholds


def compute_auc_trapezoid(fpr: np.ndarray, tpr: np.ndarray) -> float:
    """Compute area under the ROC curve using the trapezoidal rule.

    Args:
        fpr: False positive rate values, shape (n_points,).
        tpr: True positive rate values, shape (n_points,).

    Returns:
        AUC in [0, 1] (0.5 = random, 1.0 = perfect).
    """
    order      = np.argsort(fpr)
    fpr_sorted = fpr[order]
    tpr_sorted = tpr[order]
    return float(np.trapz(tpr_sorted, fpr_sorted))


# ── Plot ROC curves ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
roc_colors = ['#9E9E9E', '#2196F3', '#F44336']
roc_styles = ['-', '--', '-.']

for (name, _, _), color, style in zip(configs, roc_colors, roc_styles):
    y_prob_m         = models[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _      = roc_curve_scratch(y_test, y_prob_m)
    auc_val          = compute_auc_trapezoid(fpr, tpr)
    ax.plot(fpr, tpr, linestyle=style, color=color, linewidth=2,
            label=f'{name}  (AUC = {auc_val:.4f})')
ax.plot([0, 1], [0, 1], 'k:', linewidth=1, label='Random (AUC = 0.50)')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=11)
ax.set_ylabel('True Positive Rate (TPR / Recall)', fontsize=11)
ax.set_title('ROC Curves — All Regularisation Variants', fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Verify AUC against sklearn
y_prob_nr        = models['No Regularisation'].predict_proba(X_test)[:, 1]
fpr_sk, tpr_sk, _ = sk_roc_curve(y_test, y_prob_nr)
auc_sk           = sk_auc(fpr_sk, tpr_sk)
fpr_sc, tpr_sc, _ = roc_curve_scratch(y_test, y_prob_nr)
auc_sc           = compute_auc_trapezoid(fpr_sc, tpr_sc)
print('AUC comparison (no-reg model):')
print(f'  From-scratch : {auc_sc:.6f}')
print(f'  sklearn      : {auc_sk:.6f}')
print(f'  Difference   : {abs(auc_sc - auc_sk):.2e}  (small grid vs. exact difference expected)')

In [ ]:
# ── Ablation study: regularisation strength C vs. accuracy ───────────────────
print('Running ablation study (C sweep) ...')
ablation_rows: list[dict] = []

for C_val in C_VALUES:
    for pen in ('none', 'l2', 'l1'):
        m = LogisticRegressionScratch(
            penalty=pen, C=C_val,
            learning_rate=LEARNING_RATE, num_epochs=1500,
            tolerance=TOLERANCE, random_state=SEED,
        )
        m.fit(X_train, y_train)
        ablation_rows.append({
            'C':           C_val,
            'penalty':     pen,
            'train_acc':   round(m.score(X_train, y_train), 4),
            'test_acc':    round(m.score(X_test,  y_test),  4),
            'coef_l1':     round(float(np.sum(np.abs(m.coef_))), 4),
            'n_nonzero':   int(np.sum(np.abs(m.coef_) > 1e-4)),
        })

abl_df = pd.DataFrame(ablation_rows)
pivot  = abl_df.pivot(index='C', columns='penalty', values='test_acc')

print('\nTest accuracy by (C, penalty):')
print(pivot.round(4).to_string())

best_l2_row = abl_df[abl_df['penalty'] == 'l2'].sort_values('test_acc', ascending=False).iloc[0]
best_l1_row = abl_df[abl_df['penalty'] == 'l1'].sort_values('test_acc', ascending=False).iloc[0]
print(f'\nBest L2: C={best_l2_row["C"]}, test_acc={best_l2_row["test_acc"]:.4f}')
print(f'Best L1: C={best_l1_row["C"]}, test_acc={best_l1_row["test_acc"]:.4f}')

In [ ]:
# ── Error analysis ────────────────────────────────────────────────────────────
best_model      = models['L2 Regularisation']
y_pred_best     = best_model.predict(X_test)
y_prob_best     = best_model.predict_proba(X_test)[:, 1]
misclassified   = y_pred_best != y_test
n_errors        = misclassified.sum()

print('Error analysis (L2 model):')
print(f'  Total test samples    : {len(y_test)}')
print(f'  Misclassified         : {n_errors} ({n_errors / len(y_test):.1%})')
error_confidence = np.abs(y_prob_best[misclassified] - 0.5)
print(f'  Mean margin of errors : {error_confidence.mean():.4f}  (0 = borderline, 0.5 = confident)')
print(f'  High-conf. errors     : {(error_confidence > 0.3).sum()} (margin > 0.3)')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Score distribution
correct_probs = y_prob_best[~misclassified]
error_probs   = y_prob_best[misclassified]
axes[0].hist(correct_probs, bins=30, alpha=0.6, color='#4CAF50', density=True,
             label=f'Correct ({(~misclassified).sum()})')
axes[0].hist(error_probs,   bins=15, alpha=0.75, color='#F44336', density=True,
             label=f'Errors  ({misclassified.sum()})')
axes[0].axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='threshold=0.5')
axes[0].set_xlabel('P(y=1)')
axes[0].set_ylabel('Density')
axes[0].set_title('Score Distribution: Correct vs. Errors', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# --- Spatial view of errors
axes[1].scatter(X_test[~misclassified, 0], X_test[~misclassified, 1],
                c=[COLORS[int(c)] for c in y_test[~misclassified]],
                alpha=0.45, s=18, label='Correct')
axes[1].scatter(X_test[misclassified, 0], X_test[misclassified, 1],
                c='black', s=60, marker='x', linewidths=2, label='Errors')
axes[1].set_xlabel('x₁')
axes[1].set_ylabel('x₂')
axes[1].set_title('Misclassified Samples in Feature Space', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\nObservation: most errors cluster near the true class boundary where')
print('the two moons overlap — inherently ambiguous samples due to noise.')

# ── Final comparison table ─────────────────────────────────────────────────────
print('\nFinal results table:')
rows: list[dict] = []
for name, _, _ in configs:
    m      = models[name]
    y_pred = m.predict(X_test)
    y_prob = m.predict_proba(X_test)[:, 1]
    fpr_r, tpr_r, _ = roc_curve_scratch(y_test, y_prob)
    rows.append({
        'Model':      name,
        'Test Acc':   round(m.score(X_test, y_test), 4),
        'F1':         round(all_metrics[name]['f1'],        4),
        'Log Loss':   round(all_metrics[name]['log_loss'],  4),
        'AUC':        round(compute_auc_trapezoid(fpr_r, tpr_r), 4),
        '||coef||₁':  round(float(np.sum(np.abs(m.coef_))),  4),
        'n_nonzero':  int(np.sum(np.abs(m.coef_) > 1e-4)),
    })
final_df = pd.DataFrame(rows).set_index('Model')
print(final_df.to_string())

---
## Part 5 — Summary & Lessons Learned

### Key Takeaways

1. **Sigmoid from MLE**: The sigmoid function is not arbitrary — it emerges from fitting a Bernoulli likelihood via MLE. Binary cross-entropy is its negative log-likelihood, and gradient descent falls out by differentiating BCE with respect to **w**.

2. **Gradient simplicity**: Despite the non-linear activation, the BCE gradient $\nabla \mathcal{L} = (1/n)\mathbf{X}^\top(\hat{\mathbf{y}} - \mathbf{y})$ has the same elegant structure as the linear regression gradient from 2-01 — just with sigmoid predictions instead of linear ones.

3. **L2 vs L1 geometry**: L2 penalises large weights by adding $\lambda\|\mathbf{w}\|_2^2$ to the loss, shrinking all weights proportionally. L1 penalises $\lambda\|\mathbf{w}\|_1$, and its diamond constraint geometry pushes solutions to axis-aligned corners — producing exact zeros and sparse models.

4. **Proximal gradient for L1**: Because the L1 penalty is non-differentiable at 0, standard gradient descent cannot optimise it directly. The proximal gradient method applies a gradient step then a soft-threshold operator, which is the exact solution to the $\ell_1$-regularised sub-problem.

5. **ROC and AUC decouple threshold from performance**: A single accuracy number depends on the 0.5 threshold. The ROC curve shows model performance across all thresholds, and the AUC summarises that as a threshold-independent scalar — critical for imbalanced datasets.

### What's Next

→ **2-03 (Decision Trees & Random Forests)** introduces a completely different inductive bias: axis-aligned splits on features instead of a linear decision surface. This makes trees naturally non-linear and interpretable, at the cost of high variance — which bagging and Random Forests address.

### Going Further

- **Multiclass extension**: Logistic regression generalises to softmax regression for $K > 2$ classes — revisited in 2-06 (GLMs) as a special case of the exponential family.
- **Newton's method**: Second-order optimisation (IRLS) converges much faster than gradient descent for logistic regression; sklearn’s `lbfgs` solver uses a quasi-Newton method.
- **Online learning**: Mini-batch SGD variants (Adam, RMSProp) from Module 16 are essential when $n$ is too large for full-batch gradient descent.
- Ng, A. (2004). *Feature selection, L1 vs. L2 regularization, and rotational invariance.* ICML.